# 作业4——朴素机器翻译与局部敏感哈希（LSH）

接下来你将要实现第一个机器翻译系统，之后我们会学习局部敏感哈希的工作原理。首先导入所需函数！

如果你在本地电脑运行该笔记，记得通过 nltk 下载推特样本数据集与停用词：
nltk.download('stopwords')
nltk.download('twitter_samples')

**注意**：本次作业中的`Exercise xx`编号与`UNQ_Cx`编号**并不对应**。

### 本次作业涵盖以下内容：

- [1. 英语与法语单词的词嵌入数据](#1)
  - [1.1 生成嵌入矩阵与变换矩阵](#1-1)
      - [练习 1](#ex-01)
- [2. 翻译任务](#2)
  - [2.1 将翻译视作词嵌入的线性变换](#2-1)
      - [练习 2](#ex-02)  
      - [练习 3](#ex-03)  
      - [练习 4](#ex-04)        
  - [2.2 测试翻译效果](#2-2)
      - [练习 5](#ex-05)
      - [练习 6](#ex-06)      
- [3. 局部敏感哈希（LSH）与文档检索](#3)
  - [3.1 获取文档嵌入向量](#3-1)
      - [练习 7](#ex-07)
      - [练习 8](#ex-08)      
  - [3.2 检索推文](#3-2)
  - [3.3 使用 LSH 寻找最相似推文](#3-3)
  - [3.4 计算向量对应的哈希编号](#3-4)
      - [练习 9](#ex-09)  
  - [3.5 构建哈希表](#3-5)
      - [练习 10](#ex-10)  
  - [3.6 构建全部哈希表](#3-6)
      - [练习 11](#ex-11)

In [14]:
import pdb
import pickle
import string

import time

import gensim
import matplotlib.pyplot as plt
import nltk
import numpy as np
import scipy
import sklearn
from gensim.models import KeyedVectors
from nltk.corpus import stopwords, twitter_samples
from nltk.tokenize import TweetTokenizer

from utils import (cosine_similarity, get_dict,
                   process_tweet)
from os import getcwd

In [15]:
# add folder, tmp2, from our local workspace containing pre-downloaded corpora files to nltk's data path
filePath = f"{getcwd()}/../tmp2/"
nltk.data.path.append(filePath)

<a name="1"></a>

# 1. 英语与法语单词的词嵌入数据

编写一个实现英译法的程序。

## 数据集说明

完整的英语词嵌入数据集大小约 3.64GB，法语词嵌入数据集约 629MB。为避免 Coursera 工作空间崩溃，我们提取了本次作业所需单词的词嵌入子集。

如果你想在本地计算机运行并使用完整数据集，可以下载：
* 英语词嵌入：从 Google code archive 的 word2vec 项目获取
【查找 GoogleNews-vectors-negative300.bin.gz】(https://code.google.com/archive/p/word2vec/)
    * 需要先解压该文件。
* 法语词嵌入从该仓库下载：
[cross_lingual_text_classification](https://github.com/vjstark/crosslingual_text_classification)
    * 在终端输入下面一行命令：
    `curl -o ./wiki.multi.fr.vec https://dl.fbaipublicfiles.com/arrival/vectors/wiki.multi.fr.vec`

之后复制并运行下方代码。

```python
# Use this code to download and process the full dataset on your local computer

from gensim.models import KeyedVectors

en_embeddings = KeyedVectors.load_word2vec_format('./GoogleNews-vectors-negative300.bin', binary = True)
fr_embeddings = KeyedVectors.load_word2vec_format('./wiki.multi.fr.vec')


# loading the english to french dictionaries
en_fr_train = get_dict('en-fr.train.txt')
print('The length of the english to french training dictionary is', len(en_fr_train))
en_fr_test = get_dict('en-fr.test.txt')
print('The length of the english to french test dictionary is', len(en_fr_train))

english_set = set(en_embeddings.vocab)
french_set = set(fr_embeddings.vocab)
en_embeddings_subset = {}
fr_embeddings_subset = {}
french_words = set(en_fr_train.values())

for en_word in en_fr_train.keys():
    fr_word = en_fr_train[en_word]
    if fr_word in french_set and en_word in english_set:
        en_embeddings_subset[en_word] = en_embeddings[en_word]
        fr_embeddings_subset[fr_word] = fr_embeddings[fr_word]


for en_word in en_fr_test.keys():
    fr_word = en_fr_test[en_word]
    if fr_word in french_set and en_word in english_set:
        en_embeddings_subset[en_word] = en_embeddings[en_word]
        fr_embeddings_subset[fr_word] = fr_embeddings[fr_word]


pickle.dump( en_embeddings_subset, open( "en_embeddings.p", "wb" ) )
pickle.dump( fr_embeddings_subset, open( "fr_embeddings.p", "wb" ) )
```

#### 数据子集

如果在 Coursera 工作空间完成本次作业，我们将使用词嵌入的子集数据。

In [16]:
en_embeddings_subset = pickle.load(open("en_embeddings.p", "rb"))
fr_embeddings_subset = pickle.load(open("fr_embeddings.p", "rb"))

#### 查看数据

* en_embeddings_subset：键为英文单词，值是一个300维数组，也就是该单词对应的词嵌入向量。
```
'the': array([ 0.08007812,  0.10498047,  0.04980469,  0.0534668 , -0.06738281, ....
```

* fr_embeddings_subset：键为法语单词，值是一个300维数组，也就是该单词对应的词嵌入向量。
```
'la': array([-6.18250e-03, -9.43867e-04, -8.82648e-03,  3.24623e-02,...
```

#### 加载两个英法语单词映射字典
* 训练字典
* 测试字典

In [17]:
# loading the english to french dictionaries
en_fr_train = get_dict('en-fr.train.txt')
print('The length of the English to French training dictionary is', len(en_fr_train))
en_fr_test = get_dict('en-fr.test.txt')
print('The length of the English to French test dictionary is', len(en_fr_test))

The length of the English to French training dictionary is 5000
The length of the English to French test dictionary is 1500



#### 查看英法词典

* `en_fr_train` 是一个字典：键为英文单词，值是该单词对应的法语译文。
```
{'the': 'la',
 'and': 'et',
 'was': 'était',
 'for': 'pour',
```

* `en_fr_test` 和 `en_fr_train` 结构类似，作为测试集。我们到测试阶段再使用它。


<a name="1-1"></a>

## 1.1 生成嵌入矩阵与变换矩阵

<a name="ex-01"></a>
#### 练习 01：利用词嵌入实现英文词典到法语的翻译

接下来你需要实现函数 `get_matrices`，该函数接收已加载的数据，返回矩阵 `X` 和 `Y`。

输入参数：
- `en_fr`：英文到法语的映射字典
- `en_embeddings`：英文单词对应词嵌入的字典
- `fr_embeddings`：法语单词对应词嵌入的字典

返回结果：
- 矩阵 `X` 和矩阵 `Y`。其中 X 的每一行是一个英文单词的词嵌入向量，Y 中同一行对应的向量是该英文单词法语译文的词嵌入向量。

<div style="width:image width px; font-size:100%; text-align:center;">
<img src='X_to_Y.jpg' alt="alternate text" width="width" height="height" style="width:800px;height:200px;" /> 图 2 </div>

借助 `en_fr` 字典保证：矩阵 `X` 的第 i 行与矩阵 `Y` 的第 i 行相互对应。

**要求说明**：完善 `get_matrices()` 函数：
* 遍历 `en_fr` 字典里的所有英文单词。
* 检查该单词是否同时存在对应的英文词嵌入与法语词嵌入。

<details>
<summary>
    <font size="3" color="darkgreen"><b>提示</b></font>
</summary>
    <p>
        <ul>
            <li><a href="https://realpython.com/python-sets/#set-size-and-membership" >集合（Sets）</a>是很实用的数据结构，可用来判断元素是否属于某个集合。</li>
            <li>你可以借助 <a href="https://www.w3schools.com/python/ref_dictionary_keys.asp">keys()</a> 方法获取存在词嵌入的单词。</li>
            <li>先把向量存入列表，保证 `X` 和 `Y` 内向量顺序一一对应；再使用 <a href="https://docs.scipy.org/doc/numpy-1.13.0/reference/generated/numpy.ma.vstack.html"> np.vstack()</a> 将列表合并为 NumPy 矩阵。 </li>
            <li><a href="https://docs.scipy.org/doc/numpy/reference/generated/numpy.vstack.html">numpy.vstack</a> 会把列表中的各个元素堆叠成矩阵的行。</li>
        </ul>
    </p>

In [18]:
# UNQ_C1 (唯一单元格标识，请勿修改)
def get_matrices(en_fr, french_vecs, english_vecs):
    """
    输入：
        en_fr: 英文映射法语的字典
        french_vecs: 法语单词与其对应词嵌入向量
        english_vecs: 英文单词与其对应词嵌入向量
    输出：
        X: 矩阵，每一行（原文注释columns有误）存放英文词嵌入向量
        Y: 矩阵，对应行存放匹配的法语词嵌入向量
        R: 投影矩阵，最小化弗罗贝尼乌斯范数 ||X R - Y||^2
    """

    ### 代码起始区域（将'None'替换为你的代码）###
    # TODO: 收集成对的英/法语词嵌入，并拼接构成矩阵 X、Y。
    ### 代码结束区域 ###
    X = []
    Y = []
    for en_word, fr_word in en_fr.items():
        if en_word in english_vecs and fr_word in french_vecs:
            X.append(english_vecs[en_word])
            Y.append(french_vecs[fr_word])
    X = np.vstack(X)
    Y = np.vstack(Y)

    return X, Y

接下来我们调用 `get_matrices()` 函数，得到英文词嵌入矩阵 `X_train` 与法语词嵌入矩阵 `Y_train`，两组向量分别属于各自对应的向量空间模型。

In [19]:
# UNQ_C2 (唯一单元格标识，请勿修改)
# 无需在本单元格编写代码，但该单元格参与评分，请不要改动任何内容

# 获取训练集：
X_train, Y_train = get_matrices(
    en_fr_train, fr_embeddings_subset, en_embeddings_subset)

In [20]:
X_train.shape, Y_train.shape

((4932, 300), (4932, 300))

<a name="2"></a>

# 2. 翻译任务

<div style="width:image width:image width px; font-size:100%; text-align:center;"><img src='e_to_f.jpg' alt="alternate text" width="width" height="height" style="width:700px;height:200px;" /> 图 1 </div>

编写程序，利用词嵌入与向量空间模型实现英文单词到法语单词的翻译。

<a name="2-1"></a>
## 2.1 将翻译视作词嵌入的线性变换

基于英语与法语词嵌入字典，你需要求解变换矩阵 `R`
* 给定英文单词嵌入向量 $\mathbf{e}$，将其与矩阵相乘 $\mathbf{eR}$，得到变换后的词嵌入向量 $\mathbf{f}$。
    * $\mathbf{e}$ 和 $\mathbf{f}$ 均为**行向量**。
* 之后在法语词嵌入集合中寻找 $\mathbf{f}$ 的最近邻，选出与变换后嵌入向量相似度最高的法语单词作为翻译结果。

### 将翻译描述为最小化优化问题

求解矩阵 $\mathbf{R}$，使得下式取得最小值：

$$\arg \min _{\mathbf{R}}\| \mathbf{X R} - \mathbf{Y}\|_{F}\tag{1} $$

### 弗罗贝尼乌斯范数（Frobenius norm）

对于尺寸为 $m\times n$ 的矩阵 $A$，其弗罗贝尼乌斯范数定义为矩阵所有元素绝对值平方和的平方根：

$$\|\mathbf{A}\|_{F} \equiv \sqrt{\sum_{i=1}^{m} \sum_{j=1}^{n}\left|a_{i j}\right|^{2}}\tag{2}$$

### 实际使用的损失函数

在真实应用场景中，弗罗贝尼乌斯范数损失：
$$\| \mathbf{XR} - \mathbf{Y}\|_{F}$$

通常会替换为其平方再除以样本数量 $m$：

$$ \frac{1}{m} \|  \mathbf{X R} - \mathbf{Y} \|_{F}^{2}$$

其中 $m$ 代表样本数量（矩阵 $\mathbf{X}$ 的行数）。

* 使用该损失函数求得的最优矩阵 $\mathbf{R}$，与直接使用原始弗罗贝尼乌斯范数得到的结果完全一致。
* 取平方的原因：平方形式更容易求解梯度。
* 除以 $m$ 的原因：我们更关心**单个词嵌入对应的平均损失**，而非整个训练集的总损失。
    * 训练单词越多，训练集总损失数值会随之变大；
    取平均值后，无论训练集规模大小，都可以稳定观测平均损失水平。

##### [Optional] Detailed explanation why we use norm squared instead of the norm:
<details>
<summary>
    Click for optional details
</summary>
    <p>
        <ul>
            <li>The norm is always nonnegative (we're summing up absolute values), and so is the square. 
            <li> When we take the square of all non-negative (positive or zero) numbers, the order of the data is preserved.  
            <li> For example, if 3 > 2, 3^2 > 2^2
            <li> Using the norm or squared norm in gradient descent results in the same <i>location</i> of the minimum.
            <li> Squaring cancels the square root in the Frobenius norm formula. Because of the <a href="https://en.wikipedia.org/wiki/Chain_rule"> chain rule</a>, we would have to do more calculations if we had a square root in our expression for summation.
            <li> Dividing the function value by the positive number doesn't change the optimum of the function, for the same reason as described above.
            <li> We're interested in transforming English embedding into the French. Thus, it is more important to measure average loss per embedding than the loss for the entire dictionary (which increases as the number of words in the dictionary increases).
        </ul>
    </p>
    

<a name="ex-02"></a>

### 练习 02：实现本节描述的翻译模型

#### 第1步：计算损失函数
* 损失函数为：矩阵与其近似矩阵差值的**平方弗罗贝尼乌斯范数**，再除以训练样本数量 $m$。
* 公式：
$$ L(X, Y, R)=\frac{1}{m}\sum_{i=1}^{m} \sum_{j=1}^{n}\left( a_{i j} \right)^{2}$$

其中 $a_{i j}$ 代表矩阵 $\mathbf{XR}-\mathbf{Y}$ 第 $i$ 行、第 $j$ 列上的元素。

#### 任务要求：完善 `compute_loss()` 函数

* 将矩阵 `X` 和 `R` 相乘，得到 `Y` 的近似结果
* 计算差值矩阵 `XR - Y`
* 求出该差值矩阵的平方弗罗贝尼乌斯范数，再除以样本数量 $m$。

<details>    
<summary>
    <font size="3" color="darkgreen"><b>提示</b></font>
</summary>
<p>
<ul>
   <li>可用函数：
       <a href="https://docs.scipy.org/doc/numpy/reference/generated/numpy.dot.html">Numpy dot </a>（矩阵点乘）、
       <a href="https://docs.scipy.org/doc/numpy/reference/generated/numpy.sum.html">Numpy sum</a>（求和）、
       <a href="https://docs.scipy.org/doc/numpy/reference/generated/numpy.square.html">Numpy square</a>（逐元素平方）、
       <a href="https://docs.scipy.org/doc/numpy/reference/generated/numpy.linalg.norm.html">Numpy norm</a>（范数计算）
    </li>
   <li>注意区分：逐元素运算 和 矩阵乘法。</li>
   <li>优先使用矩阵运算实现，而非 numpy 的 norm 函数。如果你选择使用 norm 函数，务必设置好额外参数，保证得到的是**损失的平方值**，而不是原始范数。</li>

</ul>
</p>

In [21]:
# UNQ_C3 (唯一单元格标识，请勿修改)
def compute_loss(X, Y, R):
    '''
    输入：
        X: 维度为 (m,n) 的矩阵，每行是英文词嵌入向量（注释columns为文档笔误）
        Y: 维度为 (m,n) 的矩阵，每行是对应的法语词嵌入向量
        R: (n,n) 维矩阵——从英文向量空间映射到法语向量空间的变换矩阵
    输出：
        L: 标量数值，对应给定 X,Y,R 的损失函数值
    '''
    ### 代码起始区域（将'None'替换为你的代码）###
    # m 为矩阵 X 的行数
    m = X.shape[0]
    
    # TODO: 计算平均平方弗罗贝尼乌斯损失。
    loos = np.linalg.norm(X @ R - Y, 'fro') ** 2
    loss = loos / m
    ### 代码结束区域 ###
    return loss

<a name="ex-03"></a>

### 练习 03

### 第2步：计算损失函数关于变换矩阵 R 的梯度

* 求解损失函数对变换矩阵 `R` 的梯度。
* 梯度本身是一个矩阵，用于衡量：`R` 发生微小变动时，损失函数会产生多大变化。
* 梯度指明我们调整 `R` 的方向，沿着该方向更新能够降低损失。
* $m$ 代表训练样本数量（矩阵 $X$ 的行数）。
* 损失函数 $L(X,Y,R)$ 的梯度公式：

$$\frac{d}{dR}L(X,Y,R)=\frac{d}{dR}\Big(\frac{1}{m}\| X R -Y\|_{F}^{2}\Big) = \frac{2}{m}X^{T} (X R - Y)$$

**任务要求**：完善下方 `compute_gradient` 函数。

<details>
<summary>
    <font size="3" color="darkgreen"><b>Hints</b></font>
</summary>
<p>
    <ul>
    <li><a href="https://docs.scipy.org/doc/numpy/reference/generated/numpy.matrix.T.html" > Transposing in numpy </a></li>
    <li><a href="https://docs.scipy.org/doc/numpy/reference/generated/numpy.ndarray.shape.html" > Finding out the dimensions</a> of matrices in numpy </li>
    <li>Remember to use numpy.dot for matrix multiplication </li>
    </ul>
</p>
 

In [22]:
# UNQ_C4 (唯一单元格标识，请勿修改)
def compute_gradient(X, Y, R):
    '''
    输入：
        X: 维度 (m,n) 的矩阵，每行存放英文词嵌入向量（文档columns为笔误）
        Y: 维度 (m,n) 的矩阵，每行存放对应的法语词嵌入向量
        R: (n,n) 矩阵 —— 英文向量空间映射至法语向量空间的变换矩阵
    输出：
        g: (n,n) 矩阵 —— 给定 X,Y,R 时损失函数 L 的梯度
    '''
    ### 代码起始区域（将'None'替换为你的代码）###
    # m 为矩阵 X 的行数
    m = X.shape[0]

    # TODO: 计算损失函数关于 R 的梯度。
    gradient = (2/m) * X.T @ (X @ R - Y)
    ### 代码结束区域 ###
    return gradient

### 第3步：利用梯度下降算法求解最优矩阵 R

#### 梯度下降

[梯度下降](https://ml-cheatsheet.readthedocs.io/en/latest/gradient_descent.html)是一种迭代算法，用于寻找函数的最优解。
* 前文提到：损失函数关于矩阵的梯度，刻画了当矩阵中某一参数发生微小变动时，损失函数会产生多大变化。
* 梯度下降借助该信息，迭代更新矩阵 `R`，直至找到使损失最小的取值。

#### 固定迭代次数进行训练

多数场景下，我们会设置固定的训练迭代步数，而非持续迭代直到损失低于某个阈值。

##### 【可选阅读】为何采用固定迭代次数
<details>
<summary>
    <font size="3" color="darkgreen"><b>点击查看详细说明</b></font>
</summary>
<p>
<ul>
    <li>不能单纯依靠训练损失降低作为停止依据。我们真正关注的是验证集损失下降、或是验证集准确率提升。实际应用中，有人会采用**早停（early stopping）策略**：持续训练直至验证准确率开始下滑，这也是模型出现过拟合的信号。
    </li>
    <li>
    那为什么不全都使用早停策略？主要原因：经过良好正则化、训练数据规模较大的模型，性能会持续小幅提升。尤其是自然语言处理领域，模型持续训练数月依旧能获得微弱优化。这也导致很难设定一个通用的停止阈值——除非业务有外部硬性指标，否则很难界定该在哪一处终止训练。
    </li>
    <li>固定步数停止训练有一大优势：能够预估训练耗时，避免无限制训练数月。你可以在既定时间预算内追求最优效果。另一个好处是方便规划学习率调度策略：例如在训练剩余10%步数时降低学习率，训练仅剩1%步数时再次下调学习率。这类学习率策略对模型效果提升显著；但如果不知道总训练时长，很难设计对应的调度方案。
    </li>
</ul>
</p>

伪代码：
1. 计算损失函数关于矩阵 $R$ 的梯度 $g$。
2. 使用如下公式更新矩阵 $R$：
$$R_{\text{new}}= R_{\text{old}}-\alpha g$$

其中 $\alpha$ 为学习率，是一个标量数值。

#### 学习率

* 学习率（也叫步长）$\alpha$ 是一个系数，决定每一轮迭代中我们对矩阵 $R$ 的调整幅度。
* 如果调整幅度过大，步长太长，可能直接越过最优解。
* 如果每次对 $R$ 的改动很小，则需要非常多轮迭代才能逼近最优解。
* 依靠学习率 $\alpha$ 来控制每一步参数更新的幅度。
* $\alpha$ 的取值需要结合具体问题选择；本算法默认使用 `learning_rate` $=0.0003$。

<a name="ex-04"></a>

### 练习 04

#### 任务要求：实现函数 `align_embeddings()`

<details>
<summary>
    <font size="3" color="darkgreen"><b>提示</b></font>
</summary>
<p>
<ul>
    <li>每一轮迭代使用 `compute_gradient()` 函数计算梯度</li>
</ul>
</p>

In [23]:
# UNQ_C5 (唯一单元格标识，请勿修改)
def align_embeddings(X, Y, train_steps=100, learning_rate=0.0003):
    '''
    输入：
        X: (m,n) 矩阵，每行是英文单词嵌入向量
        Y: (m,n) 矩阵，每行是对应的法语单词嵌入向量
        train_steps: 正整数 —— 梯度下降迭代总步数
        learning_rate: 正数浮点数 —— 梯度下降更新步长大小
    输出：
        R: (n,n) 方阵 —— 使平方弗罗贝尼乌斯范数 ||X R -Y||^2 最小的映射矩阵
    '''
    np.random.seed(129)

    # X 的列数就是词向量维度（例如300维）
    # R 为方阵，尺寸等于词向量维度
    R = np.random.rand(X.shape[1], X.shape[1])

    for i in range(train_steps):
        if i % 25 == 0:
            print(f"loss at iteration {i} is: {compute_loss(X, Y, R):.4f}")
        ### START CODE HERE ###
        # TODO: 计算梯度并更新 R。
        gradient = compute_gradient(X, Y, R)
        R -= learning_rate * gradient
        ### END CODE HERE ###
    return R

In [24]:
# UNQ_C6 (唯一单元格标识，请勿修改)
# 无需在本单元格编写代码，但该单元格参与评分，请不要改动任何内容

# 测试你实现的函数：
np.random.seed(129)
m = 10
n = 5
X = np.random.rand(m, n)
Y = np.random.rand(m, n) * .1
R = align_embeddings(X, Y)

loss at iteration 0 is: 3.7242
loss at iteration 25 is: 3.6283
loss at iteration 50 is: 3.5350
loss at iteration 75 is: 3.4442


**Expected Output:**
```
loss at iteration 0 is: 3.7242
loss at iteration 25 is: 3.6283
loss at iteration 50 is: 3.5350
loss at iteration 75 is: 3.4442
```

## 求解变换矩阵 $\mathbf{R}$

调用函数 `align_embeddings()`，基于训练集求解变换矩阵 $\mathbf{R}$。

**注意：**下方代码单元格完整运行需要几分钟（大约3分钟）。

In [25]:
# UNQ_C7 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# You do not have to input any code in this cell, but it is relevant to grading, so please do not change anything
R_train = align_embeddings(X_train, Y_train, train_steps=400, learning_rate=0.8)

loss at iteration 0 is: 963.0146
loss at iteration 25 is: 97.8292
loss at iteration 50 is: 26.8329
loss at iteration 75 is: 9.7893
loss at iteration 100 is: 4.3776
loss at iteration 125 is: 2.3281
loss at iteration 150 is: 1.4480
loss at iteration 175 is: 1.0338
loss at iteration 200 is: 0.8251
loss at iteration 225 is: 0.7145
loss at iteration 250 is: 0.6534
loss at iteration 275 is: 0.6185
loss at iteration 300 is: 0.5981
loss at iteration 325 is: 0.5858
loss at iteration 350 is: 0.5782
loss at iteration 375 is: 0.5735


##### Expected Output

```
loss at iteration 0 is: 963.0146
loss at iteration 25 is: 97.8292
loss at iteration 50 is: 26.8329
loss at iteration 75 is: 9.7893
loss at iteration 100 is: 4.3776
loss at iteration 125 is: 2.3281
loss at iteration 150 is: 1.4480
loss at iteration 175 is: 1.0338
loss at iteration 200 is: 0.8251
loss at iteration 225 is: 0.7145
loss at iteration 250 is: 0.6534
loss at iteration 275 is: 0.6185
loss at iteration 300 is: 0.5981
loss at iteration 325 is: 0.5858
loss at iteration 350 is: 0.5782
loss at iteration 375 is: 0.5735
```

<a name="2-2"></a>

## 2.2 测试翻译效果

### k近邻算法（k-Nearest Neighbors）

[k近邻算法](https://en.wikipedia.org/wiki/K-nearest_neighbors_algorithm) 
* k-NN 是一种以向量为输入，并在数据集中找出与该向量最接近的其他向量的方法。
* 其中的“k”表示要查找的“最近邻”数量（例如，k=2 表示找出最近的两个邻居）。

### 搜索翻译嵌入向量

由于我们是用一个线性变换矩阵 $\mathbf{R}$ 来近似从英语嵌入向量到法语嵌入向量的翻译函数，因此当我们把某个特定英语单词的嵌入向量 $\mathbf{e}$ 变换到法语嵌入空间时，大多数情况下并不能精确得到某个法语单词的嵌入向量。

* 这正是 k-NN 发挥重要作用的地方！通过使用以 $\mathbf{eR}$ 为输入的 1-NN（k=1 的最近邻），我们可以在矩阵 $\mathbf{Y}$ 中搜索一个与变换后向量 $\mathbf{eR}$ 最接近的嵌入向量 $\mathbf{f}$（作为矩阵的一行）。

### 余弦相似度

向量 $u$ 和 $v$ 之间的余弦相似度，定义为两者之间夹角的余弦值。
计算公式为：

$$\cos(u,v)=\frac{u\cdot v}{\left\|u\right\|\left\|v\right\|}$$

* 当 $u$ 和 $v$ 方向相同且共线时，$\cos(u,v) = 1$。
* 当 $u$ 和 $v$ 方向完全相反时，$\cos(u,v) = -1$。
* 当 $u$ 和 $v$ 彼此正交（垂直）时，$\cos(u,v) = 0$。

#### 备注：距离与相似度大致上是相反的概念。

* 我们可以从余弦相似度推导出距离度量，但余弦相似度本身不能直接用作距离度量。
* 当余弦相似度增大（趋近于 $1$）时，两个向量之间的“距离”会减小（趋近于 $0$）。
* 我们可以将 $u$ 和 $v$ 之间的余弦距离定义为：
$$d_{\text{cos}}(u,v)=1-\cos(u,v)$$

<a name="ex-05"></a>

**练习 05**：补全函数 `nearest_neighbor()`

输入参数：
* 向量 `v`，
* 一组可能的最近邻候选集 `candidates`，
* 要查找的最近邻数量 `k`。
* 距离度量应基于余弦相似度。
* `cosine_similarity` 函数已经实现并为你导入。它接受两个向量作为参数，并返回它们之间夹角的余弦值。
* 遍历 `candidates` 中的每一行，将当前行与向量 `v` 之间的相似度结果保存到一个 Python 列表中。请注意，相似度的顺序应与 `candidates` 中行向量的顺序一致。
* 现在你可以使用 [numpy.argsort](https://docs.scipy.org/doc/numpy/reference/generated/numpy.argsort.html#numpy.argsort) 对 `candidates` 的行索引进行排序。

<details>
<summary>
    <font size="3" color="darkgreen"><b>提示</b></font>
</summary>
<p>
<ul>
    <li> numpy.argsort 将值从最小到最大（即从最负到最正）进行排序。</li>
    <li> 最接近 'v' 的候选向量应具有最高的余弦相似度。</li>
    <li> 要获取列表 'tmp' 的最后一个元素，可以使用 tmp[-1:] 这样的写法。</li>
</ul>
</p>
</details>

In [ ]:
# UNQ_C8 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
def nearest_neighbor(v, candidates, k=1):
    """
    输入参数：
      - v: 你要查找其最近邻的向量
      - candidates: 一组候选向量，我们将从中查找最近邻
      - k: 要查找的最近邻数量（top k）
    输出：
      - k_idx: 按排序顺序排列的前 k 个最近邻的索引
    """
    ### 在此处开始编写代码（将 'None' 替换为你的代码） ###
    # TODO: 计算余弦相似度，对其索引进行排序，并返回前 k 个。
    similarity_list = []
    for candidate in candidates:
        similarity = cosine_similarity(v, candidate)
        similarity_list.append(similarity)
    

    # TODO: 计算余弦相似度，对索引排序，并返回相似度最高的前 k 个。
    sorted_ids = np.argsort(similarity_list)
    k_idx = sorted_ids[-k:]

    ### 在此处结束代码 ###
    return k_idx

In [35]:
# UNQ_C9 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# You do not have to input any code in this cell, but it is relevant to grading, so please do not change anything

# Test your implementation:
v = np.array([1, 0, 1])
candidates = np.array([[1, 0, 5], [-2, 5, 3], [2, 0, 1], [6, -9, 5], [9, 9, 9]])
print(candidates[nearest_neighbor(v, candidates, 3)])

[[9 9 9]
 [1 0 5]
 [2 0 1]]


**Expected Output**:

`[[9 9 9]
 [1 0 5]
 [2 0 1]]`

### 测试你的翻译并计算准确率

<a name="ex-06"></a>
**练习 06**：
补全函数 `test_vocabulary`，该函数接收英语嵌入矩阵 $X$、法语嵌入矩阵 $Y$ 以及矩阵 $R$，并返回通过 $R$ 从 $X$ 到 $Y$ 进行翻译的准确率。

* 遍历经过变换后的英语单词嵌入向量，检查最接近的法语单词向量是否属于该英语单词的正确翻译所对应的法语单词。
* 使用 `nearest_neighbor`（参数 `k=1`）获取最近的法语嵌入向量的索引，并将其与你刚变换的英语嵌入向量所对应的索引进行比较。
* 记录获得正确翻译的次数。
* 按如下公式计算准确率：
$$\text{准确率}=\frac{\#(\text{正确预测数})}{\#(\text{总预测数})}$$

In [36]:
# UNQ_C10（唯一单元格标识，请勿修改）
def test_vocabulary(X, Y, R):
    '''
    输入：
        X：矩阵，矩阵内各列对应英文词嵌入向量（注释columns为文档笔误，实际每行是一组英文嵌入）
        Y：矩阵，矩阵内各列对应匹配的法语词嵌入向量
        R：变换矩阵，用于将词嵌入向量从英文向量空间转换至法语词嵌入向量空间
    输出：
        accuracy：英文首都词汇翻译为法语首都词汇任务的预测准确率
    '''

    ### 代码起始区域（将'None'替换为你的代码）###
    # 任务：对X执行变换，为每一行向量在Y中寻找距离最近的行，计算准确率。
    ### 代码结束区域 ###
    correct_predictions = 0
    for i in range(X.shape[0]):
        # 将英文词嵌入向量映射到法语词嵌入向量空间
        transformed_vector = X[i] @ R

        # 在Y中找到最近的法语词嵌入向量的索引
        nearest_index = nearest_neighbor(transformed_vector, Y, k=1)[0]

        # 检查预测是否正确
        if nearest_index == i:
            correct_predictions += 1

    accuracy = correct_predictions / X.shape[0]

    return accuracy

Let's see how is your translation mechanism working on the unseen data:

In [37]:
X_val, Y_val = get_matrices(en_fr_test, fr_embeddings_subset, en_embeddings_subset)

In [38]:
# UNQ_C11 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# You do not have to input any code in this cell, but it is relevant to grading, so please do not change anything

acc = test_vocabulary(X_val, Y_val, R_train)  # this might take a minute or two
print(f"accuracy on test set is {acc:.3f}")

accuracy on test set is 0.557


**Expected Output**:

```
0.557
```

You managed to translate words from one language to another language
without ever seing them with almost 56% accuracy by using some basic
linear algebra and learning a mapping of words from one language to another!

<a name="3"></a>

# 3. 局部敏感哈希（LSH）与文档检索

在本次作业的这一部分，你将借助局部敏感哈希，实现效率更高的 k 近邻算法。
随后把该算法应用到文档检索任务中。

* 处理推文数据，将每条推文转换成向量形式（使用向量嵌入表示一份文档）。
* 利用局部敏感哈希与 k 近邻算法，查找和目标推文相似的其他推文。

In [ ]:
# get the positive and negative tweets
all_positive_tweets = twitter_samples.strings('positive_tweets.json')
all_negative_tweets = twitter_samples.strings('negative_tweets.json')
all_tweets = all_positive_tweets + all_negative_tweets

In [40]:
all_tweets

['#FollowFriday @France_Inte @PKuchly57 @Milipol_Paris for being top engaged members in my community this week :)',
 '@Lamb2ja Hey James! How odd :/ Please call our Contact Centre on 02392441234 and we will be able to assist you :) Many thanks!',
 '@DespiteOfficial we had a listen last night :) As You Bleed is an amazing track. When are you in Scotland?!',
 '@97sides CONGRATS :)',
 'yeaaaah yippppy!!!  my accnt verified rqst has succeed got a blue tick mark on my fb profile :) in 15 days',
 '@BhaktisBanter @PallaviRuhail This one is irresistible :)\n#FlipkartFashionFriday http://t.co/EbZ0L2VENM',
 "We don't like to keep our lovely customers waiting for long! We hope you enjoy! Happy Friday! - LWWF :) https://t.co/smyYriipxI",
 '@Impatientraider On second thought, there’s just not enough time for a DD :) But new shorts entering system. Sheep must be buying.',
 'Jgh , but we have to go to Bayan :D bye',
 'As an act of mischievousness, am calling the ETL layer of our in-house warehousing 

<a name="3-1"></a>

### 3.1 获取文档嵌入向量

#### 词袋（BOW）文档模型
文本文档由一连串单词构成。
* 单词的先后顺序会影响语义。例如句子“Apple pie is better than pepperoni pizza.”（苹果派比意式香肠披萨更好吃）与“Pepperoni pizza is better than apple pie”（意式香肠披萨比苹果派更好吃），仅仅因为单词顺序不同，表达的含义截然相反。
* 但在部分应用场景下，忽略单词顺序，依然能够训练出高效且效果尚可的模型。
* 这种建模方式被称作词袋文档模型。

#### 文档嵌入
* 文档嵌入向量，由文档内所有单词的嵌入向量相加得到。
* 如果某个单词不存在对应的嵌入向量，我们可以直接舍弃该单词。

<a name="ex-07"></a>

**练习 07：**
完善 `get_document_embedding()` 函数。
* 函数 `get_document_embedding()` 将整篇文档编码为一份文档嵌入向量。
* 接收参数：一段文档文本（字符串格式）、字典 `en_embeddings`
* 处理文档内容，查询每个单词对应的嵌入向量。
* 将所有单词向量求和，返回这条推文处理后全部词向量相加得到的总和向量。

<details>
<summary>
    <font size="3" color="darkgreen"><b>提示</b></font>
</summary>
<p>
<ul>
    <li>相比于方括号取值方式（"[ ]"），使用Python字典的 `get()` 方法能更方便地处理不存在的单词。相关说明可以点击<a href="https://stackoverflow.com/a/11041421/12816433" >此处</a>查看</li>
    <li>未收录单词对应的默认取值应当是零向量。NumPy 在求和运算时，可以把简单的标量0进行广播，转换成零向量参与计算。</li>
    <li>另一种方案：如果单词不在字典内，则直接跳过本次累加。</li>
    <li>你可以调用 `process_tweet()` 函数来处理推文文本。该函数接收一条推文，返回分词后的单词列表。</li>
</ul>
</p>

In [41]:
# UNQ_C12（唯一单元格标识，请勿修改）
def get_document_embedding(tweet, en_embeddings): 
    '''
    输入：
        - tweet：字符串，一条推文文本
        - en_embeddings：字典，存放各个单词对应的词嵌入向量
    输出：
        - doc_embedding：推文内所有有效单词嵌入向量相加得到的文档嵌入向量
    '''
    doc_embedding = np.zeros(300)

    ### START CODE HERE（将'None'替换为你的代码）###
    # 任务：处理推文文本，把所有存在嵌入向量的单词的词向量累加起来

    # 处理推文文本
    processed_tweet = process_tweet(tweet)

    # 遍历处理后的推文中的每个单词
    for word in processed_tweet:
        # 检查单词是否在嵌入字典中
        if word in en_embeddings:
            # 如果存在，将其嵌入向量累加到文档嵌入向量中
            doc_embedding += en_embeddings[word]
        
    
    ### END CODE HERE ###
    return doc_embedding

In [42]:
# UNQ_C13 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# You do not have to input any code in this cell, but it is relevant to grading, so please do not change anything

# testing your function
custom_tweet = "RT @Twitter @chapagain Hello There! Have a great day. :) #good #morning http://chapagain.com.np"
tweet_embedding = get_document_embedding(custom_tweet, en_embeddings_subset)
tweet_embedding[-5:]

array([-0.00268555, -0.15378189, -0.55761719, -0.07216644, -0.32263184])

**Expected output**:

```
array([-0.00268555, -0.15378189, -0.55761719, -0.07216644, -0.32263184])
```

<a name="ex-08"></a>

### 练习 08

#### 将所有文档向量存入字典
接下来我们把全部推文的嵌入向量保存到字典中。
实现函数 `get_document_vecs()`

In [ ]:
# UNQ_C14（唯一单元格标识，请勿修改）
def get_document_vecs(all_docs, en_embeddings):
    '''
    输入：
        - all_docs：字符串列表 —— 数据集内全部推文文本
        - en_embeddings：字典，键为单词，值为对应单词的嵌入向量
    输出：
        - document_vec_matrix：推文嵌入向量构成的矩阵
        - ind2Doc_dict：字典，键是向量里推文对应的索引，值是推文文档嵌入向量
    '''

    # 字典的键是整型索引，用来唯一标识某一条推文
    # 字典的值是这条文档对应的文档嵌入向量
    ind2Doc_dict = {}

    # 列表，用于存放各个文档向量
    document_vec_l = []

    for i, doc in enumerate(all_docs):

        ### START CODE HERE（将'None'替换为你的代码）###
        # 任务：计算当前文档的嵌入向量，存入字典并追加到列表
        doc_vec = get_document_embedding(doc, en_embeddings)
        ind2Doc_dict[i] = doc_vec
        document_vec_l.append(doc_vec)
        ### END CODE HERE ###

    # 将文档向量列表转换成二维数组（每一行代表一条文档向量）
    document_vec_matrix = np.vstack(document_vec_l)

    return document_vec_matrix, ind2Doc_dict

In [146]:
document_vecs, ind2Tweet = get_document_vecs(all_tweets, en_embeddings_subset)

In [147]:
# UNQ_C15 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# You do not have to input any code in this cell, but it is relevant to grading, so please do not change anything

print(f"length of dictionary {len(ind2Tweet)}")
print(f"shape of document_vecs {document_vecs.shape}")

length of dictionary 10000
shape of document_vecs (10000, 300)


##### Expected Output
```
length of dictionary 10000
shape of document_vecs (10000, 300)
```

<a name="3-2"></a>

## 3.2 Looking up the tweets

Now you have a vector of dimension (m,d) where `m` is the number of tweets
(10,000) and `d` is the dimension of the embeddings (300).  Now you
will input a tweet, and use cosine similarity to see which tweet in our
corpus is similar to your tweet.

In [148]:
my_tweet = 'i am sad'
process_tweet(my_tweet)
tweet_embedding = get_document_embedding(my_tweet, en_embeddings_subset)

In [149]:
# UNQ_C16 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# You do not have to input any code in this cell, but it is relevant to grading, so please do not change anything

# this gives you a similar tweet as your input.
# this implementation is vectorized...
idx = np.argmax(cosine_similarity(document_vecs, tweet_embedding))
print(all_tweets[idx])

@zoeeylim sad sad sad kid :( it's ok I help you watch the match HAHAHAHAHA


##### Expected Output

```
@zoeeylim sad sad sad kid :( it's ok I help you watch the match HAHAHAHAHA
```

<a name="3-3"></a>

## 3.3 Finding the most similar tweets with LSH

You will now implement locality sensitive hashing (LSH) to identify the most similar tweet.
* Instead of looking at all 10,000 vectors, you can just search a subset to find
its nearest neighbors.

Let's say your data points are plotted like this:


<div style="width:image width px; font-size:100%; text-align:center;"><img src='one.png' alt="alternate text" width="width" height="height" style="width:400px;height:200px;" /> Figure 3 </div>

You can divide the vector space into regions and search within one region for nearest neighbors of a given vector.

<div style="width:image width px; font-size:100%; text-align:center;"><img src='four.png' alt="alternate text" width="width" height="height" style="width:400px;height:200px;" /> Figure 4 </div>

In [150]:
N_VECS = len(all_tweets)       # This many vectors.
N_DIMS = len(ind2Tweet[1])     # Vector dimensionality.
print(f"Number of vectors is {N_VECS} and each has {N_DIMS} dimensions.")

Number of vectors is 10000 and each has 300 dimensions.


#### Choosing the number of planes

* Each plane divides the space to $2$ parts.
* So $n$ planes divide the space into $2^{n}$ hash buckets.
* We want to organize 10,000 document vectors into buckets so that every bucket has about $~16$ vectors.
* For that we need $\frac{10000}{16}=625$ buckets.
* We're interested in $n$, number of planes, so that $2^{n}= 625$. Now, we can calculate $n=\log_{2}625 = 9.29 \approx 10$.

In [151]:
# The number of planes. We use log2(625) to have ~16 vectors/bucket.
N_PLANES = 10
# Number of times to repeat the hashing to improve the search.
N_UNIVERSES = 25

<a name="3-4"></a>

## 3.4 Getting the hash number for a vector

For each vector, we need to get a unique number associated to that vector in order to assign it to a "hash bucket".

### Hyperlanes in vector spaces
* In $3$-dimensional vector space, the hyperplane is a regular plane. In $2$ dimensional vector space, the hyperplane is a line.
* Generally, the hyperplane is subspace which has dimension $1$ lower than the original vector space has.
* A hyperplane is uniquely defined by its normal vector.
* Normal vector $n$ of the plane $\pi$ is the vector to which all vectors in the plane $\pi$ are orthogonal (perpendicular in $3$ dimensional case).

### Using Hyperplanes to split the vector space
We can use a hyperplane to split the vector space into $2$ parts.
* All vectors whose dot product with a plane's normal vector is positive are on one side of the plane.
* All vectors whose dot product with the plane's normal vector is negative are on the other side of the plane.

### Encoding hash buckets
* For a vector, we can take its dot product with all the planes, then encode this information to assign the vector to a single hash bucket.
* When the vector is pointing to the opposite side of the hyperplane than normal, encode it by 0.
* Otherwise, if the vector is on the same side as the normal vector, encode it by 1.
* If you calculate the dot product with each plane in the same order for every vector, you've encoded each vector's unique hash ID as a binary number, like [0, 1, 1, ... 0].

<a name="ex-09"></a>

### Exercise 09: Implementing hash buckets

We've initialized hash table `hashes` for you. It is list of `N_UNIVERSES` matrices, each describes its own hash table. Each matrix has `N_DIMS` rows and `N_PLANES` columns. Every column of that matrix is a `N_DIMS`-dimensional normal vector for each of `N_PLANES` hyperplanes which are used for creating buckets of the particular hash table.

*Exercise*: Your task is to complete the function `hash_value_of_vector` which places vector `v` in the correct hash bucket.

* First multiply your vector `v`, with a corresponding plane. This will give you a vector of dimension $(1,\text{N_planes})$.
* You will then convert every element in that vector to 0 or 1.
* You create a hash vector by doing the following: if the element is negative, it becomes a 0, otherwise you change it to a 1.
* You then compute the unique number for the vector by iterating over `N_PLANES`
* Then you multiply $2^i$ times the corresponding bit (0 or 1).
* You will then store that sum in the variable `hash_value`.

**Intructions:** Create a hash for the vector in the function below.
Use this formula:

$$ hash = \sum_{i=0}^{N-1} \left( 2^{i} \times h_{i} \right) $$

#### Create the sets of planes
* Create multiple (25) sets of planes (the planes that divide up the region).
* You can think of these as 25 separate ways of dividing up the vector space with a different set of planes.
* Each element of this list contains a matrix with 300 rows (the word vector have 300 dimensions), and 10 columns (there are 10 planes in each "universe").

In [152]:
np.random.seed(0)
planes_l = [np.random.normal(size=(N_DIMS, N_PLANES))
            for _ in range(N_UNIVERSES)]

<details>
<summary>
    <font size="3" color="darkgreen"><b>Hints</b></font>
</summary>
<p>
<ul>
    <li> numpy.squeeze() removes unused dimensions from an array; for instance, it converts a (10,1) 2D array into a (10,) 1D array</li>
</ul>
</p>

In [153]:
# UNQ_C17 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
def hash_value_of_vector(v, planes):
    """Create a hash for a vector; hash_id says which random hash to use.
    Input:
        - v:  vector of tweet. It's dimension is (1, N_DIMS)
        - planes: matrix of dimension (N_DIMS, N_PLANES) - the set of planes that divide up the region
    Output:
        - res: a number which is used as a hash for your vector

    """
    ### START CODE HERE (REPLACE INSTANCES OF 'None' with your code) ###
    # TODO: use the signs of v @ planes and encode the bits as sum(2**i * h_i).
    ### END CODE HERE ###

    # cast hash_value as an integer
    hash_value = int(hash_value)

    return hash_value


In [154]:
# UNQ_C18 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# You do not have to input any code in this cell, but it is relevant to grading, so please do not change anything

np.random.seed(0)
idx = 0
planes = planes_l[idx]  # get one 'universe' of planes to test the function
vec = np.random.rand(1, 300)
print(f" The hash value for this vector,",
      f"and the set of planes at index {idx},",
      f"is {hash_value_of_vector(vec, planes)}")

 The hash value for this vector, and the set of planes at index 0, is 768


##### Expected Output

```
The hash value for this vector, and the set of planes at index 0, is 768
```

<a name="3-5"></a>

## 3.5 Creating a hash table

<a name="ex-10"></a>

### Exercise 10

Given that you have a unique number for each vector (or tweet), You now want to create a hash table. You need a hash table, so that given a hash_id, you can quickly look up the corresponding vectors. This allows you to reduce your search by a significant amount of time.

<div style="width:image width px; font-size:100%; text-align:center;"><img src='table.png' alt="alternate text" width="width" height="height" style="width:500px;height:200px;" />  </div>

We have given you the `make_hash_table` function, which maps the tweet vectors to a bucket and stores the vector there. It returns the `hash_table` and the `id_table`. The `id_table` allows you know which vector in a certain bucket corresponds to what tweet.

<details>    
<summary>
    <font size="3" color="darkgreen"><b>Hints</b></font>
</summary>
<p>
<ul>
    <li> a dictionary comprehension, similar to a list comprehension, looks like this: `{i:0 for i in range(10)}`, where the key is 'i' and the value is zero for all key-value pairs. </li>
</ul>
</p>

In [155]:
# UNQ_C19 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# This is the code used to create a hash table: feel free to read over it
def make_hash_table(vecs, planes):
    """
    Input:
        - vecs: list of vectors to be hashed.
        - planes: the matrix of planes in a single "universe", with shape (embedding dimensions, number of planes).
    Output:
        - hash_table: dictionary - keys are hashes, values are lists of vectors (hash buckets)
        - id_table: dictionary - keys are hashes, values are list of vectors id's
                            (it's used to know which tweet corresponds to the hashed vector)
    """
    ### START CODE HERE (REPLACE INSTANCES OF 'None' with your code) ###

    # number of planes is the number of columns in the planes matrix
    num_of_planes = planes.shape[-1]

    # number of buckets is 2^(number of planes)
    num_buckets = 2 ** num_of_planes

    # create the hash table as a dictionary.
    # Keys are integers (0,1,2.. number of buckets)
    # Values are empty lists
    hash_table = {i:[] for i in range(num_buckets)}

    # create the id table as a dictionary.
    # Keys are integers (0,1,2... number of buckets)
    # Values are empty lists
    id_table = {i:[] for i in range(num_buckets)}

    # for each vector in 'vecs'
    for i, v in enumerate(vecs):
        # calculate the hash value for the vector
        h = hash_value_of_vector(v, planes)

        # store the vector into hash_table at key h,
        # by appending the vector v to the list at key h
        hash_table[h].append(v)

        # store the vector's index 'i' (each document is given a unique integer 0,1,2...)
        # the key is the h, and the 'i' is appended to the list at key h
        id_table[h].append(i)

    ### END CODE HERE ###

    return hash_table, id_table


In [156]:
# UNQ_C20 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# You do not have to input any code in this cell, but it is relevant to grading, so please do not change anything

np.random.seed(0)
planes = planes_l[0]  # get one 'universe' of planes to test the function
vec = np.random.rand(1, 300)
tmp_hash_table, tmp_id_table = make_hash_table(document_vecs, planes)

print(f"The hash table at key 0 has {len(tmp_hash_table[0])} document vectors")
print(f"The id table at key 0 has {len(tmp_id_table[0])}")
print(f"The first 5 document indices stored at key 0 of are {tmp_id_table[0][0:5]}")

The hash table at key 0 has 3 document vectors
The id table at key 0 has 3
The first 5 document indices stored at key 0 of are [3276, 3281, 3282]


##### Expected output
```
The hash table at key 0 has 3 document vectors
The id table at key 0 has 3
The first 5 document indices stored at key 0 of are [3276, 3281, 3282]
```

<a name="3-6"></a>

### 3.6 Creating all hash tables

You can now hash your vectors and store them in a hash table that
would allow you to quickly look up and search for similar vectors.
Run the cell below to create the hashes. By doing so, you end up having
several tables which have all the vectors. Given a vector, you then
identify the buckets in all the tables.  You can then iterate over the
buckets and consider much fewer vectors. The more buckets you use, the
more accurate your lookup will be, but also the longer it will take.

In [157]:
# Creating the hashtables
hash_tables = []
id_tables = []
for universe_id in range(N_UNIVERSES):  # there are 25 hashes
    print('working on hash universe #:', universe_id)
    planes = planes_l[universe_id]
    hash_table, id_table = make_hash_table(document_vecs, planes)
    hash_tables.append(hash_table)
    id_tables.append(id_table)

working on hash universe #: 0
working on hash universe #: 1
working on hash universe #: 2
working on hash universe #: 3
working on hash universe #: 4
working on hash universe #: 5
working on hash universe #: 6
working on hash universe #: 7
working on hash universe #: 8
working on hash universe #: 9
working on hash universe #: 10
working on hash universe #: 11
working on hash universe #: 12
working on hash universe #: 13
working on hash universe #: 14
working on hash universe #: 15
working on hash universe #: 16
working on hash universe #: 17
working on hash universe #: 18
working on hash universe #: 19
working on hash universe #: 20
working on hash universe #: 21
working on hash universe #: 22
working on hash universe #: 23
working on hash universe #: 24


### Approximate K-NN

<a name="ex-11"></a>

### Exercise 11

Implement approximate K nearest neighbors using locality sensitive hashing,
to search for documents that are similar to a given document at the
index `doc_id`.

##### Inputs
* `doc_id` is the index into the document list `all_tweets`.
* `v` is the document vector for the tweet in `all_tweets` at index `doc_id`.
* `planes_l` is the list of planes (the global variable created earlier).
* `k` is the number of nearest neighbors to search for.
* `num_universes_to_use`: to save time, we can use fewer than the total
number of available universes.  By default, it's set to `N_UNIVERSES`,
which is $25$ for this assignment.

The `approximate_knn` function finds a subset of candidate vectors that
are in the same "hash bucket" as the input vector 'v'.  Then it performs
the usual k-nearest neighbors search on this subset (instead of searching
through all 10,000 tweets).

<details>
<summary>
    <font size="3" color="darkgreen"><b>Hints</b></font>
</summary>
<p>
<ul>
    <li> There are many dictionaries used in this function.  Try to print out planes_l, hash_tables, id_tables to understand how they are structured, what the keys represent, and what the values contain.</li>
    <li> To remove an item from a list, use `.remove()` </li>
    <li> To append to a list, use `.append()` </li>
    <li> To add to a set, use `.add()` </li>
</ul>
</p>

In [158]:
# UNQ_C21 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# This is the code used to do the fast nearest neighbor search. Feel free to go over it
def approximate_knn(doc_id, v, planes_l, k=1, num_universes_to_use=N_UNIVERSES):
    """Search for k-NN using hashes."""
    assert num_universes_to_use <= N_UNIVERSES

    # Vectors that will be checked as possible nearest neighbor
    vecs_to_consider_l = list()

    # list of document IDs
    ids_to_consider_l = list()

    # create a set for ids to consider, for faster checking if a document ID already exists in the set
    ids_to_consider_set = set()

    # loop through the universes of planes
    for universe_id in range(num_universes_to_use):

        # get the set of planes from the planes_l list, for this particular universe_id
        planes = planes_l[universe_id]

        # get the hash value of the vector for this set of planes
        hash_value = hash_value_of_vector(v, planes)

        # get the hash table for this particular universe_id
        hash_table = hash_tables[universe_id]

        # get the list of document vectors for this hash table, where the key is the hash_value
        document_vectors_l = hash_table[hash_value]

        # get the id_table for this particular universe_id
        id_table = id_tables[universe_id]

        # get the subset of documents to consider as nearest neighbors from this id_table dictionary
        new_ids_to_consider = id_table[hash_value]

        ### START CODE HERE (REPLACE INSTANCES OF 'None' with your code) ###

        # TODO: remove doc_id and collect unique candidate vectors and IDs from this bucket.

        ### END CODE HERE ###

    # Now run k-NN on the smaller set of vecs-to-consider.
    print("Fast considering %d vecs" % len(vecs_to_consider_l))

    # convert the vecs to consider set to a list, then to a numpy array
    vecs_to_consider_arr = np.array(vecs_to_consider_l)

    # call nearest neighbors on the reduced list of candidate vectors
    nearest_neighbor_idx_l = nearest_neighbor(v, vecs_to_consider_arr, k=k)

    # Use the nearest neighbor index list as indices into the ids to consider
    # create a list of nearest neighbors by the document ids
    nearest_neighbor_ids = [ids_to_consider_l[idx]
                            for idx in nearest_neighbor_idx_l]

    return nearest_neighbor_ids


In [159]:
#document_vecs, ind2Tweet
doc_id = 0
doc_to_search = all_tweets[doc_id]
vec_to_search = document_vecs[doc_id]

In [160]:
# UNQ_C22 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# You do not have to input any code in this cell, but it is relevant to grading, so please do not change anything

# Sample
nearest_neighbor_ids = approximate_knn(
    doc_id, vec_to_search, planes_l, k=3, num_universes_to_use=5)

removed doc_id 0 of input vector from new_ids_to_search
removed doc_id 0 of input vector from new_ids_to_search
removed doc_id 0 of input vector from new_ids_to_search
removed doc_id 0 of input vector from new_ids_to_search
removed doc_id 0 of input vector from new_ids_to_search
Fast considering 77 vecs


In [161]:
print(f"Nearest neighbors for document {doc_id}")
print(f"Document contents: {doc_to_search}")
print("")

for neighbor_id in nearest_neighbor_ids:
    print(f"Nearest neighbor at document id {neighbor_id}")
    print(f"document contents: {all_tweets[neighbor_id]}")

Nearest neighbors for document 0
Document contents: #FollowFriday @France_Inte @PKuchly57 @Milipol_Paris for being top engaged members in my community this week :)

Nearest neighbor at document id 2140
document contents: @PopsRamjet come one, every now and then is not so bad :)
Nearest neighbor at document id 701
document contents: With the top cutie of Bohol :) https://t.co/Jh7F6U46UB
Nearest neighbor at document id 51
document contents: #FollowFriday @France_Espana @reglisse_menthe @CCI_inter for being top engaged members in my community this week :)


# 4 Conclusion
Congratulations - Now you can look up vectors that are similar to the
encoding of your tweet using LSH!

# 中文题干速查

下面是本作业各道 Exercise 的中文翻译，保留英文原题方便对照。

## Exercise 01：构造英法词向量矩阵
根据英法词典，遍历所有英文-法文词对，只保留两边都有词向量的词对。将英文向量按顺序组成矩阵 `X`，将对应法文向量按同样顺序组成矩阵 `Y`。

## Exercise 02：计算损失
实现 `compute_loss()`：计算 `XR`，求 `XR - Y`，将差矩阵的所有元素平方后求和，再除以训练样本数 `m`。

## Exercise 03：计算梯度
实现 `compute_gradient()`，根据公式 `2/m * X.T @ (X @ R - Y)` 计算损失函数对 `R` 的梯度。

## Exercise 04：用梯度下降对齐向量空间
实现 `align_embeddings()`：随机初始化变换矩阵 `R`，重复计算梯度并更新 `R`，使 `XR` 尽量接近 `Y`。

## Exercise 05：寻找最近邻
实现 `nearest_neighbor()`：计算输入向量 `v` 与每个候选向量的余弦相似度，找出相似度最高的前 `k` 个候选，并返回它们的下标。

## Exercise 06：测试翻译准确率
实现 `test_vocabulary()`：把每个英文向量乘以 `R`，在法文矩阵 `Y` 中找到最近邻。如果最近邻的下标与正确法文向量的下标一致，就记为预测正确，最后计算准确率。

## Exercise 07：生成文档向量
实现 `get_document_embedding()`：处理 tweet，把其中所有已知词的词向量相加，得到整条 tweet 的文档向量；未知词可以忽略。

## Exercise 08：保存所有文档向量
实现 `get_document_vecs()`：遍历所有 tweet，生成每条 tweet 的文档向量，保存文档下标到向量的字典，并把所有向量堆叠成二维矩阵。

## Exercise 09：计算哈希桶编号
实现 `hash_value_of_vector()`：计算向量与所有平面的点积，将负数编码为 0、非负数编码为 1，再用 `Σ(2**i * h_i)` 把 bit 编码成一个整数桶编号。

## Exercise 10：创建哈希表
题目已提供 `make_hash_table()`，它把向量放入对应桶，并记录每个桶中的 tweet 下标。此部分保留课程提供的代码。

## Exercise 11：实现近似 KNN
实现 `approximate_knn()`：在多个哈希空间中找到与输入向量落入同一桶的候选 tweet，合并并去重、排除输入 tweet 自身，然后只在候选集合上执行普通 KNN。